# Load Historical Auction Data (Web UI Export)

Use this notebook to load and persist `historical_auction_values.csv` into a separate Iceberg namespace (`yhnfl_manual`).

In [13]:
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 0 ns (started: 2026-07-26 14:53:07 -05:00)


In [14]:
from pathlib import Path
import os

import polars as pl
from nfl.yahoo_fantasy import (
    YahooApiClient,
    YahooWarehouseClient,
    build_oauth_session,
    configure_polars_display,
    load_historical_auction_values,
    persist_historical_auction_tables,
    resolve_historical_players,
)

time: 0 ns (started: 2026-07-26 14:53:07 -05:00)


In [15]:
configure_polars_display(rows=50, cols=20, str_length=100)
client = YahooWarehouseClient.from_project_root()
_ = client.ensure_registered()
project_root = client.paths.project_root
os.chdir(project_root)
print('Project root:', project_root)
print('Catalog URI:', client.paths.catalog_uri)

Project root: C:\Users\EricTruett\nfl
Catalog URI: sqlite:///C:/Users/EricTruett/nfl/iceberg_catalog.db
time: 0 ns (started: 2026-07-26 14:53:07 -05:00)


In [16]:
CSV_PATH = project_root / 'historical_auction_values.csv'
MIN_CONFIDENCE = 0.97
REVIEW_CONFIDENCE = 0.90
WRITE_HISTORICAL_TABLES = True
TARGET_NAMESPACE = 'yhnfl_manual'

# Optional: fetch additional Yahoo player pools across historical seasons
FETCH_EXTRA_PLAYER_POOL = True
EXTRA_POOL_START_SEASON = 2020
EXTRA_POOL_END_SEASON = 2025
WRITE_EXTRA_PLAYER_POOL = True
EXTRA_PLAYER_POOL_PATH = project_root / 'output' / 'historical_auction_extra_player_pool.parquet'

print('CSV path:', CSV_PATH)
print('Write enabled:', WRITE_HISTORICAL_TABLES)
print('Target namespace:', TARGET_NAMESPACE)
print('Fetch extra player pool:', FETCH_EXTRA_PLAYER_POOL)
print('Season range:', f"{EXTRA_POOL_START_SEASON}-{EXTRA_POOL_END_SEASON}")
print('Write extra player pool:', WRITE_EXTRA_PLAYER_POOL)
print('Extra player pool path:', EXTRA_PLAYER_POOL_PATH)

CSV path: C:\Users\EricTruett\nfl\historical_auction_values.csv
Write enabled: True
Target namespace: yhnfl_manual
Fetch extra player pool: True
Season range: 2020-2025
Write extra player pool: True
Extra player pool path: C:\Users\EricTruett\nfl\output\historical_auction_extra_player_pool.parquet
time: 15 ms (started: 2026-07-26 14:53:07 -05:00)


In [20]:
extra_player_df = pl.DataFrame()
base_player_df = client.load_table('yahoo_common.player')

if FETCH_EXTRA_PLAYER_POOL:
    required_env_vars = [
        'YAHOO_CLIENT_ID',
        'YAHOO_CLIENT_SECRET',
        'YAHOO_REDIRECT_URI',
    ]
    missing_env_vars = [name for name in required_env_vars if not os.environ.get(name)]

    if missing_env_vars:
        print('Skipping extra Yahoo player pool fetch. Missing env vars:', missing_env_vars)
    else:
        auth_code = os.environ.get('YAHOO_AUTH_CODE')
        token_candidates = [
            project_root / '.yahoo_token.json',
            project_root / '.secrets' / 'yahoo_token.json',
        ]
        token_path = next((p for p in token_candidates if p.exists()), token_candidates[0])
        print('Using Yahoo token path:', token_path)
        try:
            oauth = build_oauth_session(
                client_id=os.environ['YAHOO_CLIENT_ID'],
                client_secret=os.environ['YAHOO_CLIENT_SECRET'],
                redirect_uri=os.environ['YAHOO_REDIRECT_URI'],
                token_path=token_path,
                auth_code=auth_code,
                open_browser=False,
            )

            # Reload to pick up latest local source changes without kernel restart.
            import importlib
            import nfl.yahoo_fantasy.api as yahoo_api_module
            importlib.reload(yahoo_api_module)
            api = yahoo_api_module.YahooApiClient(oauth_session=oauth, use_cache=True)

            try:
                if hasattr(api, 'get_players_for_season_range'):
                    extra_rows = api.get_players_for_season_range(
                        start_season=EXTRA_POOL_START_SEASON,
                        end_season=EXTRA_POOL_END_SEASON,
                        sport='nfl',
                    )
                elif hasattr(api, 'get_players_for_season'):
                    extra_rows = []
                    for season in range(EXTRA_POOL_START_SEASON, EXTRA_POOL_END_SEASON + 1):
                        try:
                            extra_rows.extend(api.get_players_for_season(season=season, sport='nfl'))
                        except ValueError:
                            continue
                    dedup = {}
                    for row in extra_rows:
                        dedup[str(row.get('player_key') or '')] = row
                    extra_rows = [r for k, r in dedup.items() if k]
                else:
                    extra_rows = []
                    print('Skipping extra pool fetch: YahooApiClient does not expose season-based player APIs in this runtime.')

                extra_player_df = pl.DataFrame(extra_rows) if extra_rows else pl.DataFrame()
                print('Fetched extra Yahoo player rows:', extra_player_df.height)
            except ValueError as exc:
                print('No extra Yahoo player rows found for configured season range.')
                print(str(exc))
        except ValueError as exc:
            print('Skipping extra Yahoo player pool fetch due to OAuth setup issue.')
            print(str(exc))
            print("Tip: set YAHOO_AUTH_CODE once or run scripts/regenerate_yahoo_token.py to seed token cache")
        except RuntimeError as exc:
            print('Skipping extra Yahoo player pool fetch due to OAuth token exchange failure.')
            print(str(exc))

if WRITE_EXTRA_PLAYER_POOL and extra_player_df.height > 0:
    EXTRA_PLAYER_POOL_PATH.parent.mkdir(parents=True, exist_ok=True)
    extra_player_df.write_parquet(EXTRA_PLAYER_POOL_PATH)
    print('Wrote extra player pool to:', EXTRA_PLAYER_POOL_PATH)

if extra_player_df.height > 0:
    player_df = pl.concat([base_player_df, extra_player_df], how='diagonal_relaxed').unique(
        subset=['player_key'], keep='last'
    )
else:
    player_df = base_player_df

print('Player pool summary')
print('  base rows:', base_player_df.height)
print('  extra rows:', extra_player_df.height)
print('  combined rows:', player_df.height)

Using Yahoo token path: C:\Users\EricTruett\nfl\.secrets\yahoo_token.json
Fetched extra Yahoo player rows: 16401
Wrote extra player pool to: C:\Users\EricTruett\nfl\output\historical_auction_extra_player_pool.parquet
Player pool summary
  base rows: 3459
  extra rows: 16401
  combined rows: 16401
time: 1min 24s (started: 2026-07-26 14:54:27 -05:00)


In [29]:
# Reload to ensure notebook uses latest local matcher logic.
import importlib
import nfl.yahoo_fantasy.historical_auction as historical_auction_module
importlib.reload(historical_auction_module)

historical_raw = historical_auction_module.load_historical_auction_values(CSV_PATH)

historical_import = historical_auction_module.resolve_historical_players(
    raw_df=historical_raw,
    yahoo_player_df=player_df,
    min_confidence=MIN_CONFIDENCE,
    review_confidence=REVIEW_CONFIDENCE,
)

resolved_df = historical_import.resolved
queue_df = historical_import.match_queue

print('Import summary')
print('  raw rows:', historical_raw.height)
print('  resolved rows:', resolved_df.filter(pl.col('resolution_status') == 'resolved').height)
print('  ambiguous rows:', resolved_df.filter(pl.col('resolution_status') == 'ambiguous').height)
print('  unresolved rows:', resolved_df.filter(pl.col('resolution_status') == 'unresolved').height)
print('  queue rows:', queue_df.height)

Import summary
  raw rows: 705
  resolved rows: 705
  ambiguous rows: 0
  unresolved rows: 0
  queue rows: 0
time: 1.44 s (started: 2026-07-26 15:08:25 -05:00)


In [30]:
if WRITE_HISTORICAL_TABLES:
    persist_report = persist_historical_auction_tables(
        import_result=historical_import,
        namespace=TARGET_NAMESPACE,
        dry_run=False,
        replace_existing=True,
    )
    print('Historical tables persisted:')
    for row in persist_report:
        print(f'  {row.table_identifier}: {row.rows_written} rows')
else:
    print('Write skipped. Set WRITE_HISTORICAL_TABLES = True to persist tables.')

Historical tables persisted:
  yhnfl_manual.historical_auction_values_raw: 705 rows
  yhnfl_manual.historical_auction_values_resolved: 705 rows
  yhnfl_manual.historical_auction_values_match_queue: 0 rows
time: 141 ms (started: 2026-07-26 15:08:29 -05:00)


In [31]:
if queue_df.height == 0:
    print('No unresolved rows.')
else:
    queue_preview = queue_df.sort(['resolution_status', 'resolution_confidence', 'season'], descending=[False, True, False])
    print(f'Queue rows needing review: {queue_preview.height}')
    print(queue_preview.head(100))

No unresolved rows.
time: 0 ns (started: 2026-07-26 15:08:29 -05:00)
